# Linear conjugate-gradient inversion using Sep Vectors

### Preliminary steps

Importing necessary libraries

In [1]:
import sys,os,imp
sys.path.insert(0,"/net/server/homes/sep/ettore/research/packages/pySolver/GenericSolver/python")
sys.path.append("/net/server/homes/sep/ettore/research/packages/acoustic_isotropic_operators/local/lib/python")
import genericIO
SepVector=genericIO.SepVector
import numpy as np
import pyOperator as Op
import pyProblem as Prblm
import pyStopperBase as Stopper
import pyLCGsolver as LCG
import pyVector

Defining matrix-vector multiplicatication operation

In [2]:
class MatMult_SepVector(Op.Operator):
	"""Operator class to perform matrix-vector multiplication"""

	def __init__(self,A,domain,range):
		"""Constructor for the class: A = matrix to use; domain = domain vector; range = range vector"""
		if(not isinstance(domain,SepVector.vector)): raise TypeError("ERROR! Domain vector not a Vector object")
		if(not isinstance(range,SepVector.vector)): raise TypeError("ERROR! Range vector not a Vector object")
		#Setting domain and range of operator and matrix to use during application of the operator
		self.setDomainRange(domain,range)
		self.A = np.matrix(A)
		return

	def forward(self,add,model,data):
		"""Method to compute d = A m"""
		self.checkDomainRange(model,data)
		if(not isinstance(model,SepVector.vector)): raise TypeError("ERROR! Model vector not a Vector object")
		if(not isinstance(data,SepVector.vector)): raise TypeError("ERROR! Data vector not a Vector object")
		if(not add): data.zero()
		#Converting to numpy arrays
		data_np=data.getNdArray()
		model_np=model.getNdArray()
		data_np+=np.matmul(A,model_np)
		return

	def adjoint(self,add,model,data):
		"""Method to compute m = A d"""
		self.checkDomainRange(model,data)
		if(not isinstance(model,SepVector.vector)): raise TypeError("ERROR! Model vector not a Vector object")
		if(not isinstance(data,SepVector.vector)): raise TypeError("ERROR! Data vector not a Vector object")
		if(not add): model.zero()
		#Converting to numpy arrays
		data_np=data.getNdArray()
		model_np=model.getNdArray()
		model_np+=np.matmul(A.H,data_np)
		return

### Operator and Vector instantiations

In [3]:
#Create a sepVector
nsamp=200
model=SepVector.getSepVector(ns=[1,nsamp],storage="dataDouble")
data=SepVector.getSepVector(ns=[1,nsamp],storage="dataDouble")
A = np.matrix(np.zeros((nsamp,nsamp),dtype=np.float64))
np.fill_diagonal(A, -2)
np.fill_diagonal(A[1:], 1)
np.fill_diagonal(A[:,1:], 1)
#Create operator
MatMultSym = MatMult_SepVector(A,model,data)
#Filling data with ones
data.set(1.)

### Inversion definition 

Instantiating the following problem:

$\phi(\mathbf{m}) = \frac{1}{2} \|A\mathbf{m}-\mathbf{d} \|_2^2$

In [4]:
#Create L2-norm linear problem
L2Prob_sym = Prblm.ProblemL2Linear(model,data,MatMultSym)

Creating linear conjugate-gradient solver

In [5]:
#Create stopper
niter = 10000
Stop  = Stopper.BasicStopper(niter=niter)
#Create solver
LCGsolver = LCG.LCGsolver(Stop)
#Setting saving parameters
LCGsolver.setDefaults(iter_sampling=100,save_obj=True,iter_buffer_size=100,save_model=True,prefix=None)

Showing documentation

In [6]:
help(LCGsolver)

Help on LCGsolver in module pyLCGsolver object:

class LCGsolver(pySolver.Solver)
 |  Linear-Conjugate Gradient and Steepest-Descent Solver parent object
 |  
 |  Method resolution order:
 |      LCGsolver
 |      pySolver.Solver
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __del__(self)
 |      Default destructor
 |  
 |  __init__(self, stoppr, steepest=False, logger=None)
 |      Constructor for LCG Solver
 |  
 |  run(self, prblm, verbose=True, restart=False)
 |      Running LCG and steppest-descent solver
 |  
 |  ----------------------------------------------------------------------
 |  Methods inherited from pySolver.Solver:
 |  
 |  save_results(self, iter, prblm, force_save=False, force_write=False)
 |      Method to save results
 |      force_saving = [False]; Flag to ignore iteration sampling
 |      force_write  = [False]; Force writing on disk if necessary (used to handle last iteration)
 |  
 |  setDefaults(self, save_obj=False, save_res=False, save_grad

Running the solver

In [7]:
LCGsolver.run(L2Prob_sym)

LINEAR CONJUGATE GRADIENT SOLVER
Restart folder: /net/storm/scr1/ettore/restart_2019-01-28T11-52-50.237852/

iter = 0 obj = 100.0 residual norm = 14.142135620117188 gradient norm= 1.4142135381698608 feval = 2
iter = 1 obj = 99.8 residual norm = 14.127986907958984 gradient norm= 1.4142135381698608 feval = 3
iter = 2 obj = 99.64829396325459 residual norm = 14.117244720458984 gradient norm= 1.1661903858184814 feval = 5
iter = 3 obj = 99.52078939803243 residual norm = 14.108209609985352 gradient norm= 1.0497703552246094 feval = 7
iter = 4 obj = 99.40857155873738 residual norm = 14.100253105163574 gradient norm= 0.9757096767425537 feval = 9
iter = 5 obj = 99.30715308864362 residual norm = 14.093058586120605 gradient norm= 0.9223321676254272 feval = 11
iter = 6 obj = 99.21389408350707 residual norm = 14.086440086364746 gradient norm= 0.8810795545578003 feval = 13
iter = 7 obj = 99.12708427153905 residual norm = 14.080275535583496 gradient norm= 0.8477320075035095 feval = 15
iter = 8 obj = 99

iter = 78 obj = 96.11174151460135 residual norm = 13.864468574523926 gradient norm= 0.31275618076324463 feval = 157
iter = 79 obj = 96.04507038730152 residual norm = 13.859658241271973 gradient norm= 0.2929398715496063 feval = 159
iter = 80 obj = 95.97098177945767 residual norm = 13.8543119430542 gradient norm= 0.2738322913646698 feval = 161
iter = 81 obj = 95.88849190047105 residual norm = 13.848356246948242 gradient norm= 0.2553860545158386 feval = 163
iter = 82 obj = 95.7976542273883 residual norm = 13.841795921325684 gradient norm= 0.23758922517299652 feval = 165
iter = 83 obj = 95.79150438346684 residual norm = 13.841351509094238 gradient norm= 0.2929370105266571 feval = 167
iter = 84 obj = 95.69088363034041 residual norm = 13.83407974243164 gradient norm= 0.32311853766441345 feval = 169
iter = 85 obj = 95.67487452017917 residual norm = 13.83292293548584 gradient norm= 0.2127540558576584 feval = 171
iter = 86 obj = 95.57410127852015 residual norm = 13.82563591003418 gradient norm=

iter = 158 obj = 92.04463093304588 residual norm = 13.567950248718262 gradient norm= 1.0892771482467651 feval = 317
iter = 159 obj = 91.9787090690152 residual norm = 13.563090324401855 gradient norm= 0.8549949526786804 feval = 319
iter = 160 obj = 91.94967397241987 residual norm = 13.560949325561523 gradient norm= 0.5800836682319641 feval = 321
iter = 161 obj = 91.93771393463024 residual norm = 13.560067176818848 gradient norm= 0.38850605487823486 feval = 323
iter = 162 obj = 91.93088502278675 residual norm = 13.559563636779785 gradient norm= 0.2490580826997757 feval = 325
iter = 163 obj = 91.9284498544715 residual norm = 13.5593843460083 gradient norm= 0.17809119820594788 feval = 327
iter = 164 obj = 91.92709513036255 residual norm = 13.559284210205078 gradient norm= 0.11939235776662827 feval = 329
iter = 165 obj = 91.92619671585106 residual norm = 13.55921745300293 gradient norm= 0.09028488397598267 feval = 331
iter = 166 obj = 91.92522162277018 residual norm = 13.5591459274292 gradi

iter = 300 obj = 78.92662032246866 residual norm = 12.563965797424316 gradient norm= 0.027653541415929794 feval = 601
iter = 301 obj = 78.926554643225 residual norm = 12.563960075378418 gradient norm= 0.029097266495227814 feval = 603
iter = 302 obj = 78.92653927809317 residual norm = 12.563959121704102 gradient norm= 0.01512300968170166 feval = 605
iter = 303 obj = 78.92652743658954 residual norm = 12.563959121704102 gradient norm= 0.008814395405352116 feval = 607
iter = 304 obj = 78.92649166061031 residual norm = 12.563955307006836 gradient norm= 0.012578925117850304 feval = 609
iter = 305 obj = 78.92644944403284 residual norm = 12.563952445983887 gradient norm= 0.01875988580286503 feval = 611
iter = 306 obj = 78.9264321455536 residual norm = 12.563950538635254 gradient norm= 0.014883347786962986 feval = 613
iter = 307 obj = 78.92642081462716 residual norm = 12.563950538635254 gradient norm= 0.00977905374020338 feval = 615
iter = 308 obj = 78.92639737978774 residual norm = 12.56394863

iter = 375 obj = 1.9503916873845975 residual norm = 1.975040078163147 gradient norm= 1.272373914718628 feval = 751
iter = 376 obj = 1.1544111500062706 residual norm = 1.5194809436798096 gradient norm= 1.4020822048187256 feval = 753
iter = 377 obj = 0.840405450925991 residual norm = 1.2964608669281006 gradient norm= 1.4343819618225098 feval = 755
iter = 378 obj = 0.7453547160950644 residual norm = 1.2209460735321045 gradient norm= 0.9196516275405884 feval = 757
iter = 379 obj = 0.7149842639857364 residual norm = 1.1958129405975342 gradient norm= 0.6868664622306824 feval = 759
iter = 380 obj = 0.6650485525717922 residual norm = 1.1532983779907227 gradient norm= 0.5310370922088623 feval = 761
iter = 381 obj = 0.540846620612254 residual norm = 1.040044903755188 gradient norm= 0.48119667172431946 feval = 763
iter = 382 obj = 0.4750839419024686 residual norm = 0.9747655391693115 gradient norm= 0.7628905177116394 feval = 765
iter = 383 obj = 0.23635875742191934 residual norm = 0.6875445842742

...

iter = 593 obj = 3.073698564793976e-14 residual norm = 2.479394538568158e-07 gradient norm= 9.14453082145883e-08 feval = 1187
iter = 594 obj = 2.932324858415742e-14 residual norm = 2.42170386854923e-07 gradient norm= 1.343156412758617e-07 feval = 1189
iter = 595 obj = 2.7312672278769255e-14 residual norm = 2.3372065527382802e-07 gradient norm= 7.972661109079127e-08 feval = 1191
iter = 596 obj = 2.2588605012890306e-14 residual norm = 2.1254930970826535e-07 gradient norm= 1.683169728039502e-07 feval = 1193
iter = 597 obj = 1.984796585112846e-14 residual norm = 1.9923837157875823e-07 gradient norm= 1.6388615620144265e-07 feval = 1195
iter = 598 obj = 1.3922891282561399e-14 residual norm = 1.668705635893275e-07 gradient norm= 1.6502734467849223e-07 feval = 1197
iter = 599 obj = 1.1557914160112482e-14 residual norm = 1.520389076858919e-07 gradient norm= 1.9777230875206442e-07 feval = 1199
iter = 600 obj = 1.1213533848121801e-14 residual norm = 1.4975670126204932e-07 gradient norm= 6.0780187

iter = 662 obj = 2.379353618670431e-16 residual norm = 2.1814461348412806e-08 gradient norm= 3.6742264786226997e-09 feval = 1325
iter = 663 obj = 2.328358049368601e-16 residual norm = 2.157942624592124e-08 gradient norm= 4.323402968964274e-09 feval = 1327
iter = 664 obj = 2.2658462049419304e-16 residual norm = 2.128777154553063e-08 gradient norm= 7.909233268321714e-09 feval = 1329
iter = 665 obj = 2.236750488204934e-16 residual norm = 2.1150651008383647e-08 gradient norm= 6.472821389991168e-09 feval = 1331
iter = 666 obj = 2.225566774217465e-16 residual norm = 2.1097710245499002e-08 gradient norm= 2.392653630778341e-09 feval = 1333
iter = 667 obj = 2.1368520179362673e-16 residual norm = 2.0672938916277417e-08 gradient norm= 3.2084916945507302e-09 feval = 1335
iter = 668 obj = 2.0894117064332707e-16 residual norm = 2.044217062291409e-08 gradient norm= 6.706901700681556e-09 feval = 1337
iter = 669 obj = 2.052406088388813e-16 residual norm = 2.0260335631405724e-08 gradient norm= 6.0856542

iter = 739 obj = 1.158865556168693e-18 residual norm = 1.5224096427957079e-09 gradient norm= 1.5110749318480998e-09 feval = 1479
iter = 740 obj = 1.015421814049513e-18 residual norm = 1.4250767232937278e-09 gradient norm= 1.3758050254608634e-09 feval = 1481
iter = 741 obj = 8.747683407900598e-19 residual norm = 1.3227005046800855e-09 gradient norm= 9.110250887012228e-10 feval = 1483
iter = 742 obj = 7.566377180773254e-19 residual norm = 1.2301526464142398e-09 gradient norm= 1.1045666603592963e-09 feval = 1485
iter = 743 obj = 5.002304690802863e-19 residual norm = 1.0002304540179807e-09 gradient norm= 1.0246213877351806e-09 feval = 1487
iter = 744 obj = 4.419002650160513e-19 residual norm = 9.401066591863128e-10 gradient norm= 9.71126845605852e-10 feval = 1489
iter = 745 obj = 4.293555844112325e-19 residual norm = 9.266666878282592e-10 gradient norm= 3.588578600943748e-10 feval = 1491
iter = 746 obj = 4.121612953882516e-19 residual norm = 9.079221263696979e-10 gradient norm= 3.660428904

iter = 817 obj = 1.1314851449285017e-20 residual norm = 1.5043172263418114e-10 gradient norm= 4.687339058562223e-11 feval = 1635
iter = 818 obj = 1.0963055231658572e-20 residual norm = 1.480746775195385e-10 gradient norm= 4.183306478999782e-11 feval = 1637
iter = 819 obj = 1.0573430623408034e-20 residual norm = 1.4541960691172306e-10 gradient norm= 3.873213902383377e-11 feval = 1639
iter = 820 obj = 9.48341141891238e-21 residual norm = 1.3772008533585733e-10 gradient norm= 8.036954435297616e-11 feval = 1641
iter = 821 obj = 7.165957147753149e-21 residual norm = 1.1971597324578198e-10 gradient norm= 1.166676477648565e-10 feval = 1643
iter = 822 obj = 6.682003180060405e-21 residual norm = 1.1560279816746899e-10 gradient norm= 6.895539000906226e-11 feval = 1645
iter = 823 obj = 6.46092453577174e-21 residual norm = 1.1367431301811948e-10 gradient norm= 5.139820166699671e-11 feval = 1647
iter = 824 obj = 6.251022847977365e-21 residual norm = 1.1181254533365603e-10 gradient norm= 4.515847765

iter = 889 obj = 2.4487710754500876e-21 residual norm = 6.998244345135518e-11 gradient norm= 3.022284896306293e-12 feval = 1779
iter = 890 obj = 2.448107857897237e-21 residual norm = 6.997296492228244e-11 gradient norm= 2.5099609890499863e-12 feval = 1781
iter = 891 obj = 2.4456734991205546e-21 residual norm = 6.993816636935435e-11 gradient norm= 2.588507966999609e-12 feval = 1783
iter = 892 obj = 2.4426631717146154e-21 residual norm = 6.98951105326806e-11 gradient norm= 4.962356480592289e-12 feval = 1785
iter = 893 obj = 2.4415323729891022e-21 residual norm = 6.987892903209669e-11 gradient norm= 3.3589912535464395e-12 feval = 1787
iter = 894 obj = 2.4411970835374756e-21 residual norm = 6.987412731751519e-11 gradient norm= 1.9047684436668266e-12 feval = 1789
iter = 895 obj = 2.440288308140094e-21 residual norm = 6.986112383033927e-11 gradient norm= 2.022061988335433e-12 feval = 1791
iter = 896 obj = 2.4400069342743198e-21 residual norm = 6.98570923329811e-11 gradient norm= 2.1741328355